In [ ]:
%%configure -f
{
  "driverMemory": "28g",
  "executorMemory": "28g",
  "conf": {
    "spark.microsoft.delta.parquet.vorder.enabled": "true",
    "spark.databricks.delta.optimizeWrite.enabled": "true",
    "spark.sql.adaptive.enabled": "true",
    "spark.native.enabled": "true"
  }
}

# Silver Layer - Data Cleaning and Transformation

This notebook cleans the raw data in the Bronze layer, applies business logic, and persists the curated data in the Silver layer.

**Actions:**
- Deduplication (using latest ingestion timestamp)
- Handling nulls
- Standardizing formats
- Persisting into curated Silver Delta tables

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Identify bronze table
bronze_table = "bronze.raw_invoices"
try:
    bronze_df = spark.table(bronze_table)
except Exception as e:
    # Fallback to schema-less name
    bronze_table = "bronze_raw_invoices"
    bronze_df = spark.table(bronze_table)

print(f"Reading from: {bronze_table}")

In [ ]:
# 1. Deduplication: Take only the latest ingestion for each invoice
window_spec = Window.partitionBy("invoice_id").orderBy(F.desc("ingestion_timestamp"))

dedup_df = bronze_df.withColumn("rn", F.row_number().over(window_spec)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

print(f"Records after deduplication: {dedup_df.count()}")

In [ ]:
# 2. Data Cleaning & Transformation
silver_df = dedup_df.withColumn(
    "quantity", F.coalesce(F.col("quantity"), F.lit(1))
).withColumn(
    "unit_price", F.coalesce(F.col("unit_price"), F.lit(0.0))
).withColumn(
    "discount", F.coalesce(F.col("discount"), F.lit(0.0))
).withColumn(
    "calculated_revenue", 
    (F.col("unit_price") * F.col("quantity")) * (1.0 - F.col("discount"))
)

print("Transformation applied successfully.")

In [ ]:
# 3. Write to Silver layer
def lakehouse_is_schema_enabled():
    import uuid
    probe = f"_schema_probe_{uuid.uuid4().hex}"
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {probe}")
        spark.sql(f"DROP SCHEMA IF EXISTS {probe}")
        return True
    except Exception as e:
        if "Feature not supported on Apache Spark in Microsoft Fabric" in str(e):
            return False
        raise

schema_enabled = lakehouse_is_schema_enabled()

if schema_enabled:
    spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
    silver_table = "silver.curated_invoices"
else:
    silver_table = "silver_curated_invoices"

# Use Liquid Clustering (if available) for optimized analytical queries
try:
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(silver_table)
    
    spark.sql(f"ALTER TABLE {silver_table} CLUSTER BY (region, product_category)")
    spark.sql(f"OPTIMIZE {silver_table}")
except Exception as e:
    print(f"Standard write: {e}")
    silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"✅ Silver table written: {silver_table}")